In [8]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model

In [10]:
# Set the path to your dataset
dataset_path = 'CNN letter Dataset'

# Image dimensions
img_width, img_height = 75, 100

In [11]:
# Data generators for training and validation (Fixed rescaling)
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(img_width, img_height),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(img_width, img_height),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

# Print class indices to confirm correct label mapping
print("Class indices:", train_generator.class_indices)

Found 28403 images belonging to 35 classes.
Found 7100 images belonging to 35 classes.
Class indices: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9, 'A': 10, 'B': 11, 'C': 12, 'D': 13, 'E': 14, 'F': 15, 'G': 16, 'H': 17, 'I': 18, 'J': 19, 'K': 20, 'L': 21, 'M': 22, 'N': 23, 'P': 24, 'Q': 25, 'R': 26, 'S': 27, 'T': 28, 'U': 29, 'V': 30, 'W': 31, 'X': 32, 'Y': 33, 'Z': 34}


In [12]:

# Define the CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_width, img_height, 3)),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(train_generator.num_classes, activation='softmax')
])

In [13]:
# Compile the model
model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size,
    epochs=20
)

Epoch 1/20
887/887 ━━━━━━━━━━━━━━━━━━━━ 69s 76ms/step - accuracy: 0.4922 - loss: 1.8221 - val_accuracy: 0.9662 - val_loss: 0.1371
Epoch 2/20
887/887 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.9375 - loss: 0.1732 - val_accuracy: 0.9672 - val_loss: 0.1369
Epoch 3/20
887/887 ━━━━━━━━━━━━━━━━━━━━ 21s 18ms/step - accuracy: 0.9106 - loss: 0.2734 - val_accuracy: 0.9808 - val_loss: 0.0758
Epoch 4/20
887/887 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9688 - loss: 0.1369 - val_accuracy: 0.9816 - val_loss: 0.0772
Epoch 5/20
887/887 ━━━━━━━━━━━━━━━━━━━━ 17s 18ms/step - accuracy: 0.9482 - loss: 0.1574 - val_accuracy: 0.9883 - val_loss: 0.0477
Epoch 6/20
887/887 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9688 - loss: 0.0901 - val_accuracy: 0.9881 - val_loss: 0.0486
Epoch 7/20
887/887 ━━━━━━━━━━━━━━━━━━━━ 18s 18ms/step - accuracy: 0.9600 - loss: 0.1191 - val_accuracy: 0.9864 - val_loss: 0.0577
Epoch 8/20
887/887 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9062 - loss: 0.1281 - val_accu

In [27]:
model.save('ocr_cnn_model.h5')
print("Model saved successfully!")

Model saved successfully!


In [28]:
# Evaluate the model on the validation set
loss, accuracy = model.evaluate(validation_generator, steps=validation_generator.samples // validation_generator.batch_size)
print(f'Validation accuracy: {accuracy * 100:.2f}%')

221/221 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.9918 - loss: 0.0246
Validation accuracy: 99.26%


In [10]:
# Replace 'your_model.h5' with the path to your .h5 file
model = load_model('ocr_cnn_model.h5')

In [22]:
# Function to preprocess test image
def load_and_preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(img_width, img_height), color_mode='rgb')
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0  # Normalize the image
    return img_array

# Path to the image you want to test
test_image_path = 'dirty_h_resized.png'

# Load and preprocess the image
test_image = load_and_preprocess_image(test_image_path)

# Predict the class of the image
predictions = model.predict(test_image)
predicted_class = np.argmax(predictions, axis=1)

# Map the predicted class index to the class label
class_labels = list(train_generator.class_indices.keys())
predicted_label = class_labels[predicted_class[0]]

print(f'The predicted class is: {predicted_label}')



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
The predicted class is: 6
